# Clínica - Projeto Itaú

In [ ]:
import json
import os
import re
import unicodedata
import random
import pandas as pd
import asyncio
import hashlib
import time
from pathlib import Path

import nest_asyncio

Entrada e normalização
Leitura do CSV e ajuste inicial de texto/codificação.

In [ ]:
# LINHA DE SELEÇÃO DO INPUT
df = pd.read_csv("dataset_clinica_20261.csv", encoding="utf-8") # 4s para carregar

def corrigir_mojibake(valor):
    if isinstance(valor, str) and ("Ã" in valor or "Â" in valor):
        try:
            return valor.encode("latin1").decode("utf-8")
        except UnicodeError:
            return valor
    return valor

colunas_texto = df.select_dtypes(include="object").columns
df[colunas_texto] = df[colunas_texto].apply(lambda col: col.map(corrigir_mojibake))

if "magistrado" in df.columns:
    df["magistrado"] = df["magistrado"].astype(str).str.strip().str.upper()

display(df.head())
df.info()

Preparação da amostra para análise com IA

100 casos aleatórios (reproduzíveis via `random_state=42`).

In [ ]:
df_curto = df.copy()
df_curto = df_curto.sample(n=100, random_state=42).reset_index(drop=True)

In [ ]:
df_curto.shape

Variáveis para extração

- justiça_gratuita (sim/não)
- rito_processual (Juizado Especial / Procedimento Comum)
- tipo_acao (fraude/golpe, cobrança indevida, empréstimo não reconhecido, revisão contratual)
- contato_previo_banco (sim/não)
- canal_contato (SAC, Ouvidoria, Procon, Reclame Aqui, Agência, não identificado)
- mencao_reclame_aqui (sim/não)
- boletim_de_ocorrencia (sim/não)
- resultado_julgamento (procedente, improcedente, parcialmente procedente, extinto)
- culpa_atribuida (banco, consumidor, terceiro, compartilhada)
- valor_danos_morais (float, R$)
- valor_danos_materiais (float, R$)


## Extração com OpenAI — async
Extrai as variáveis em JSON com cache e retry.

In [ ]:
nest_asyncio.apply()

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Defina OPENAI_API_KEY antes de executar esta célula.")

async_client = AsyncOpenAI(api_key=api_key)
modelo_openai = "gpt-4.1-mini"
MAX_CONCORRENTE = 20
SALVAR_CACHE_A_CADA = 50

# --- Cache local ---
CACHE_PATH = Path("cache") / "cache_openai.json"
CACHE_PATH.parent.mkdir(exist_ok=True)

In [ ]:
def _carregar_cache():
    if CACHE_PATH.exists():
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}

def _salvar_cache(cache):
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

cache_local = _carregar_cache()
print(f"Cache carregado: {len(cache_local)} entradas existentes")

# --- Prompts ---
prompt_sistema = (
    "Você é um analista jurídico especializado em processos cíveis contra bancos. "
    "Extraia informações estruturadas de decisões judiciais em português. "
    "Responda APENAS com JSON válido, sem texto adicional."
)

template_prompt = """Analise o documento judicial abaixo e extraia as variáveis indicadas.

PASSO 1 — TRIAGEM:
Identifique o tipo do documento:
- "sentença": decisão que resolve o processo (total ou parcialmente)
- "despacho": ato de impulso processual sem conteúdo decisório
- "decisão interlocutória": decisão que não encerra o processo
- "outro": embargos de declaração, cumprimento de sentença, etc.

Se tipo_documento != "sentença", retorne APENAS:
{{"tipo_documento": "<valor>", "fora_do_escopo": true}}

PASSO 2 — EXTRAÇÃO (apenas para sentenças):

VARIÁVEIS:
1. tipo_documento: "sentença"
2. fora_do_escopo: false
3. justica_gratuita: ("sim" / "não" / "não identificado")
   Retorne "sim" se o texto mencionar: "Justiça Gratuita", "gratuidade da justiça",
   "gratuidade processual", "benefícios da AJG", "benefícios da assistência judiciária",
   ou se o cabeçalho listar "Justiça Gratuita" como campo do processo.
   Retorne "não" apenas se o texto mencionar explicitamente que a gratuidade foi
   indeferida ou que a parte recolheu custas.
   Retorne "não identificado" se não houver qualquer menção ao tema.
4. rito_processual: ("Juizado Especial" / "Procedimento Comum")
5. tipo_acao:
   - "empréstimo não reconhecido": autor nega ter contratado empréstimo consignado
   - "fraude em conta ou cartão": golpe, transação não autorizada em conta/cartão (motoboy, falsa central, boa noite Cinderela, etc.)
   - "revisão contratual": autor questiona taxas ou cláusulas de contrato que reconhece ter firmado
   - "cobrança indevida": cobrança de tarifa, serviço ou parcela não contratada
   - "superendividamento": pedido de repactuação com base na Lei 14.181/2021
   - "fora_do_escopo": cumprimento de sentença, extinção por inércia, acordo homologado, litispendência
   - "outro": não se encaixa em nenhuma categoria acima
6. contato_previo_banco: ("sim" / "não")
   IMPORTANTE: Este campo aceita APENAS "sim" ou "não". Nunca retorne "não identificado".
   Retorne "sim" APENAS se a petição inicial menciona que o autor contatou
   o banco ANTES de decidir processar, com objetivo de reclamar ou resolver o problema
   (ex: ligou para SAC, foi à agência, registrou no Procon).
   NÃO contar: contato feito para registrar a fraude já ocorrida, para tentar
   estornar valor após o golpe, ou qualquer contato posterior ao evento danoso.
   Em todos os outros casos (incluindo quando não há menção), retorne "não".
7. canal_contato: ("SAC" / "Ouvidoria" / "Procon" / "Reclame Aqui" / "Agência" / "não identificado")
   Retorne apenas se mencionado explicitamente no texto como canal de contato prévio.
8. mencao_reclame_aqui: ("sim" / "não")
9. boletim_de_ocorrencia: ("sim" / "não")
10. resultado_julgamento:
    - "procedente"
    - "parcialmente procedente"
    - "improcedente"
    - "extinto sem mérito": extinção por inércia, desistência, falta de pressuposto, litispendência
    - "extinto com mérito prescrição": extinção por prescrição ou decadência (art. 487 II)
    - "extinto acordo": homologação de transação entre as partes
11. culpa_atribuida: ("banco" / "consumidor" / "terceiro" / "compartilhada" / "não identificado")
    REGRA:
    - improcedente → "consumidor" (salvo exceção explícita no texto)
    - procedente ou parcialmente procedente → "banco" (salvo exceção explícita)
    - extinto (qualquer tipo) → "não identificado"
12. valor_danos_morais: número em reais
    0.0 = não condenado | -1.0 = condenado mas valor a apurar em liquidação
13. valor_danos_materiais: número em reais
    0.0 = não condenado | -1.0 = condenado mas valor a apurar em liquidação
14. repetição_indébito: ("simples" / "dobro" / "não aplicável" / "não identificado")
    Refere-se à devolução de valores cobrados indevidamente (art. 42 CDC).

REGRAS GERAIS:
- Retorne "não" ou "não identificado" quando não houver menção explícita no texto.
- Nunca infira o que não está escrito.
- Quando a sentença julgar múltiplos contratos com resultados diferentes,
  classifique pelo resultado majoritário ou use "parcialmente procedente".

Retorne exatamente este JSON:
{{"tipo_documento":"sentença","fora_do_escopo":false,"justica_gratuita":"...","rito_processual":"...","tipo_acao":"...","contato_previo_banco":"...","canal_contato":"...","mencao_reclame_aqui":"...","boletim_de_ocorrencia":"...","resultado_julgamento":"...","culpa_atribuida":"...","valor_danos_morais":0.0,"valor_danos_materiais":0.0,"repetição_indébito":"..."}}

Documento:
{decisao}"""

_campos_padrao = {
    "tipo_documento": "não identificado",
    "fora_do_escopo": False,
    "justica_gratuita": "não",
    "rito_processual": "não identificado",
    "tipo_acao": "não identificado",
    "contato_previo_banco": "não",
    "canal_contato": "não identificado",
    "mencao_reclame_aqui": "não",
    "boletim_de_ocorrencia": "não",
    "resultado_julgamento": "não identificado",
    "culpa_atribuida": "não identificado",
    "valor_danos_morais": 0.0,
    "valor_danos_materiais": 0.0,
    "repetição_indébito": "não identificado",
}

def _hash_decisao(texto):
    chave = texto[:500].strip()
    return hashlib.md5(chave.encode("utf-8")).hexdigest()

def _parse_json_seguro(conteudo):
    try:
        return json.loads(conteudo)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", conteudo, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise

async def _chamar_decisao_async(idx, texto, semaforo, max_tentativas=4):
    prompt = template_prompt.format(decisao=texto)
    async with semaforo:
        for tentativa in range(max_tentativas):
            try:
                params = dict(
                    model=modelo_openai,
                    messages=[
                        {"role": "system", "content": prompt_sistema},
                        {"role": "user", "content": prompt},
                    ],
                    response_format={"type": "json_object"},
                )
                if not modelo_openai.startswith("o"):
                    params["temperature"] = 0

                resposta = await async_client.chat.completions.create(**params)
                dados = _parse_json_seguro(resposta.choices[0].message.content.strip())

                for campo in ("valor_danos_morais", "valor_danos_materiais"):
                    try:
                        dados[campo] = float(dados.get(campo) or 0.0)
                    except (ValueError, TypeError):
                        dados[campo] = 0.0

                return idx, {**_campos_padrao, **dados}

            except Exception as e:
                is_429 = "429" in str(e)
                is_last = tentativa == max_tentativas - 1
                if is_429 and not is_last:
                    espera = min(2 ** tentativa * 5, 60)
                    jitter = random.uniform(0, espera * 0.3)
                    await asyncio.sleep(espera + jitter)
                    continue
                return idx, {**_campos_padrao, "resultado_julgamento": f"ERRO: {str(e)}"}


async def processar_async(indices_api, decisoes, resultados_final):
    semaforo = asyncio.Semaphore(MAX_CONCORRENTE)
    tasks = [_chamar_decisao_async(i, decisoes[i], semaforo) for i in indices_api]
    pendentes_flush = 0

    with tqdm(total=len(tasks), desc="Extraindo (async)", unit="dec") as barra:
        for coro in asyncio.as_completed(tasks):
            idx, resultado = await coro
            resultados_final[idx] = resultado

            if not str(resultado.get("resultado_julgamento", "")).startswith("ERRO"):
                cache_local[_hash_decisao(decisoes[idx])] = resultado
                pendentes_flush += 1

            if pendentes_flush >= SALVAR_CACHE_A_CADA:
                _salvar_cache(cache_local)
                pendentes_flush = 0

            barra.update(1)

    if pendentes_flush > 0:
        _salvar_cache(cache_local)


# --- Separa cached dos que precisam de API (reprocessa entradas com ERRO) ---
decisoes = df_curto["decisao"].fillna("").tolist()
resultados_final = [None] * len(decisoes)

indices_api = []
for i, texto in enumerate(decisoes):
    hash_key = _hash_decisao(texto)
    resultado_em_cache = cache_local.get(hash_key)

    if resultado_em_cache is None:
        indices_api.append(i)
    elif str(resultado_em_cache.get("resultado_julgamento", "")).startswith("ERRO"):
        # Remove do cache para forçar nova chamada na próxima execução
        indices_api.append(i)
        del cache_local[hash_key]
    else:
        resultados_final[i] = resultado_em_cache

hits_cache = len(decisoes) - len(indices_api)
print(f"Cache hits: {hits_cache}/{len(decisoes)} | A processar: {len(indices_api)} | Concorrência: {MAX_CONCORRENTE}")

# --- Dispara tudo assincronamente ---
t_inicio = time.time()
await processar_async(indices_api, decisoes, resultados_final)
tempo_total = time.time() - t_inicio

print(f"\nConcluído em {tempo_total:.1f}s | Cache: {len(cache_local)}/{len(decisoes)}")
print(f"Referência síncrona: 7min 22s (442s) → speedup: {442/tempo_total:.1f}×")

# --- Monta df_curto: colunas originais + campos da IA com sufixo _ia ---
colunas_originais = [c for c in df_curto.columns if not c.endswith("_ia")]
df_curto = df_curto[colunas_originais]

df_ai = pd.DataFrame(resultados_final).rename(
    columns={c: f"{c}_ia" for c in _campos_padrao.keys()}
)

df_curto = pd.concat([df_curto.reset_index(drop=True), df_ai.reset_index(drop=True)], axis=1)

print(f"\nColunas finais ({len(df_curto.columns)}): {list(df_curto.columns)}")
df_curto.head(5)

Reprocessamento dos erros

In [ ]:
## Reprocessamento

# Identifica linhas com ERRO no resultado
mask_erros = df_curto['resultado_julgamento_ia'].astype(str).str.startswith('ERRO', na=False)
indices_erro = df_curto[mask_erros].index.tolist()
print(f"Linhas com ERRO para reprocessar: {len(indices_erro)}")

if indices_erro:
    # Pega os textos e limpa o cache dessas entradas (força nova chamada)
    decisoes_erro = df_curto.loc[indices_erro, 'decisao'].fillna('').tolist()
    for texto in decisoes_erro:
        hash_key = _hash_decisao(texto)
        if hash_key in cache_local:
            del cache_local[hash_key]

    # Reprocessa só os erros
    resultados_reprocess = [None] * len(decisoes_erro)
    indices_todos = list(range(len(decisoes_erro)))

    t_inicio = time.time()
    await processar_async(indices_todos, decisoes_erro, resultados_reprocess)
    tempo_total = time.time() - t_inicio
    print(f"Reprocessamento concluído em {tempo_total:.1f}s")

    # Atualiza df_curto com os novos resultados
    df_novos = pd.DataFrame(resultados_reprocess).rename(
        columns={c: f"{c}_ia" for c in _campos_padrao.keys()}
    )

    colunas_ia = [f"{c}_ia" for c in _campos_padrao.keys()]
    df_curto.loc[indices_erro, colunas_ia] = df_novos[colunas_ia].values

    # Verifica se ainda restam erros
    erros_restantes = df_curto['resultado_julgamento_ia'].astype(str).str.startswith('ERRO', na=False).sum()
    print(f"Erros restantes após reprocessamento: {erros_restantes}")
    pasta_saida = "output"
    os.makedirs(pasta_saida, exist_ok=True)
    # Salva o arquivo atualizado
    arquivo_saida = os.path.join(pasta_saida, "curto_ia.xlsx")
    df_curto.to_excel(arquivo_saida, index=False, sheet_name="Processos")
    print("Arquivo atualizado salvo: output/curto_ia.xlsx")
else:
    print("Nenhum erro encontrado — nada a reprocessar.")

## Classificação por Regex — baseline de comparação

Para cada variável extraída pela IA, criamos um classificador por regex aplicado ao texto bruto da decisão.
Objetivo: medir o **acordo IA vs regex**. Alta concordância → regex é suficiente. Baixa concordância → IA agrega valor real.

In [ ]:
## Classificação por Regex — baseline de comparação
# ── Normalização base ──────────────────────────────────────────────────────────
def _norm(texto: str) -> str:
    """Remove acentos, lowercase, colapsa espaços."""
    if pd.isna(texto):
        return ""
    t = unicodedata.normalize("NFKD", str(texto).lower()).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", t)


# ── 1. tipo_documento ──────────────────────────────────────────────────────────
_re_sentenca      = re.compile(r"\bsentenca\b", re.IGNORECASE)
_re_despacho      = re.compile(r"\bdespacho\b", re.IGNORECASE)
_re_decisao_inter = re.compile(r"\bdecisao\s+interlocutoria\b", re.IGNORECASE)
_re_embargos      = re.compile(r"\bembargos\s+de\s+declaracao\b", re.IGNORECASE)
_re_cumprimento   = re.compile(r"\bcumprimento\s+de\s+sentenca\b", re.IGNORECASE)

def regex_tipo_documento(t: str) -> str:
    n = _norm(t)
    # Verifica o cabeçalho (primeiros 300 chars) para priorizar
    cabecalho = n[:300]
    if _re_embargos.search(cabecalho):   return "outro"
    if _re_cumprimento.search(cabecalho): return "outro"
    if _re_sentenca.search(cabecalho):   return "sentença"
    if _re_despacho.search(cabecalho):   return "despacho"
    if _re_decisao_inter.search(cabecalho): return "decisão interlocutória"
    # Fallback: busca no texto todo
    if _re_sentenca.search(n):           return "sentença"
    if _re_despacho.search(n):           return "despacho"
    return "não identificado"


# ── 2. fora_do_escopo ──────────────────────────────────────────────────────────
_re_fora = re.compile(
    r"\b(cumprimento\s+de\s+sentenca|embargos\s+de\s+declaracao"
    r"|cancelamento\s+da\s+distribuicao|cancelar\s+a\s+distribuicao"
    r"|art\.\s*924|satisf[ae]ita\s+a\s+obrigacao)\b", re.IGNORECASE
)

def regex_fora_do_escopo(t: str) -> bool:
    return bool(_re_fora.search(_norm(t)))


# ── 3. justica_gratuita ────────────────────────────────────────────────────────
_re_jg_sim = re.compile(
    r"\b(justica\s+gratuita|gratuidade\s+da\s+justica|gratuidade\s+processual"
    r"|beneficios\s+da\s+(assistencia\s+judiciaria\s+gratuita|justica\s+gratuita)"
    r"|\bajg\b|assistencia\s+judiciaria\s+gratuita)", re.IGNORECASE
)
_re_jg_nao = re.compile(
    r"\b(indefiro\s+.{0,30}gratuidade|gratuidade\s+.{0,30}indeferida"
    r"|recolhimento\s+das\s+custas\s+iniciais|sem\s+beneficio\s+da\s+gratuidade)\b",
    re.IGNORECASE
)

def regex_justica_gratuita(t: str) -> str:
    n = _norm(t)
    if _re_jg_nao.search(n): return "não"
    if _re_jg_sim.search(n): return "sim"
    return "não identificado"


# ── 4. rito_processual ─────────────────────────────────────────────────────────
_re_juizado = re.compile(r"\b(juizado\s+especial|lei\s*(n[o°º]?\s*)?9\.099)\b", re.IGNORECASE)
_re_comum   = re.compile(r"\b(procedimento\s+comum(civel)?|rito\s+ordinario)\b", re.IGNORECASE)

def regex_rito_processual(t: str) -> str:
    n = _norm(t)
    if _re_juizado.search(n): return "Juizado Especial"
    if _re_comum.search(n):   return "Procedimento Comum"
    return "não identificado"


# ── 5. tipo_acao ───────────────────────────────────────────────────────────────
# Ordem importa: mais específico → mais genérico
_re_superend = re.compile(
    r"\b(superendividamento|lei\s*(n[o°º]?\s*)?14\.181|repactuacao\s+de\s+dividas)\b",
    re.IGNORECASE
)
_re_revisao = re.compile(
    r"\b(revisao\s+(contratual|de\s+juros|de\s+contrato)|juros\s+(remuneratorios|abusivos)"
    r"|capitalizacao|anatocismo|cet\s+abusivo|tabela\s+price|taxa\s+de\s+juros\s+abusiv)\b",
    re.IGNORECASE
)
_re_emprest = re.compile(
    r"\b(emprestimo\s+(nao\s+)?(solicitado|reconhecido|autorizado|contratado)"
    r"|contrato\s+nao\s+reconhecido|descontos?\s+nao\s+(autorizados?|reconhecidos?)"
    r"|nao\s+(contratou|realizou|firmou|celebrou)\s+.{0,20}(emprestimo|contrato)"
    r"|vittima\s+de\s+fraude.{0,50}emprestimo|fraude\s+no\s+emprestimo)\b",
    re.IGNORECASE
)
_re_fraude = re.compile(
    r"\b(vitima\s+de\s+(fraude|golpe|estelionato)|transac[ao]\w*\s+fraudulent\w*"
    r"|operac[ao]\w*\s+fraudulent\w*|fraude\s+bancar\w*|sofreu\s+(fraude|golpe)"
    r"|clonac[ao]\w*\s+de\s+cart[ao]|phishing|motoboy|boa\s+noite\s+cinderela"
    r"|falsa\s+central|golpe\s+do\s+(presente|buque|buquê))\b",
    re.IGNORECASE
)
_re_cobranca = re.compile(
    r"\b(cobranca\s+indevida|desconto\s+indevido|tarifa\s+indevida"
    r"|lancamento\s+indevido|cobrado\s+indevidamente|valor\s+cobrado\s+a\s+maior"
    r"|cobranca\s+nao\s+autorizada)\b",
    re.IGNORECASE
)
_re_fora_acao = re.compile(
    r"\b(cumprimento\s+de\s+sentenca|embargos\s+de\s+declaracao"
    r"|cancelamento\s+da\s+distribuicao|litispendencia|coisa\s+julgada)\b",
    re.IGNORECASE
)

def regex_tipo_acao(t: str) -> str:
    n = _norm(t)
    if _re_fora_acao.search(n):  return "fora_do_escopo"
    if _re_superend.search(n):   return "superendividamento"
    if _re_revisao.search(n):    return "revisão contratual"
    if _re_emprest.search(n):    return "empréstimo não reconhecido"
    if _re_fraude.search(n):     return "fraude em conta ou cartão"
    if _re_cobranca.search(n):   return "cobrança indevida"
    return "outro"


# ── 6. contato_previo_banco ────────────────────────────────────────────────────
_re_contato = re.compile(
    r"\b(sac|ouvidoria|procon|reclame\s+aqui"
    r"|reclamac[ao]\s+(administrativa|junto\s+ao?|no\s+banco|na\s+instituicao)"
    r"|protocolo\s+de\s+(atendimento|reclamac[ao])"
    r"|procurou\s+o\s+banco|acionou\s+o\s+banco|comunicou\s+(ao|o)\s+banco"
    r"|tentou\s+(resolver|solucionar)\s+(junto\s+ao?\s+banco|o\s+problema\s+administrativamente)"
    r"|contato\s+administrativo|via\s+administrativa)\b",
    re.IGNORECASE
)

def regex_contato_previo(t: str) -> str:
    return "sim" if _re_contato.search(_norm(t)) else "não"


# ── 7. canal_contato ───────────────────────────────────────────────────────────
_re_reclame   = re.compile(r"\breclame\s+aqui\b", re.IGNORECASE)
_re_procon    = re.compile(r"\b(procon|decon)\b", re.IGNORECASE)
_re_ouvidoria = re.compile(r"\bouvidoria\b", re.IGNORECASE)
_re_sac       = re.compile(r"\b(sac|servico\s+de\s+atendimento\s+ao\s+consumidor)\b", re.IGNORECASE)
_re_agencia   = re.compile(
    r"(compareceu|dirigiu.se|foi\s+(ate|a\s+uma?)|presencialmente\s+na"
    r"|atendimento\s+presencial)\s+(a\s+)?agencia", re.IGNORECASE
)

def regex_canal_contato(t: str) -> str:
    n = _norm(t)
    if _re_reclame.search(n):   return "Reclame Aqui"
    if _re_procon.search(n):    return "Procon"
    if _re_ouvidoria.search(n): return "Ouvidoria"
    if _re_sac.search(n):       return "SAC"
    if _re_agencia.search(n):   return "Agência"
    return "não identificado"


# ── 8. mencao_reclame_aqui ─────────────────────────────────────────────────────
def regex_mencao_reclame(t: str) -> str:
    return "sim" if _re_reclame.search(_norm(t)) else "não"


# ── 9. boletim_de_ocorrencia ───────────────────────────────────────────────────
_re_bo = re.compile(
    r"\b(boletim\s+de\s+ocorrencia|registro\s+(de\s+)?(ocorrencia|policial)"
    r"|B\.O\.\s*n[o°º]?|b\.o\.\s+n[o°º]?)\b",
    re.IGNORECASE
)

def regex_boletim(t: str) -> str:
    return "sim" if _re_bo.search(_norm(t)) else "não"


# ── 10. resultado_julgamento ───────────────────────────────────────────────────
def regex_resultado(t: str) -> str:
    n = _norm(t)
    # Busca trecho após JULGO ou HOMOLOGO
    m = re.search(r"\b(?:julgo|homologo)\b(.{0,300})", n)
    trecho = m.group(1) if m else n[:300]

    if re.search(r"parcialmente\s+procedente", trecho):
        return "parcialmente procedente"
    if re.search(r"improcedent(?:e|es)", trecho):
        return "improcedente"
    if re.search(r"procedent(?:e|es)", trecho):
        return "procedente"
    # Extinções — ordem: mais específico primeiro
    if re.search(r"prescric[ao]|decadenci", trecho):
        return "extinto com mérito prescrição"
    if re.search(r"homologo\s+o\s+acordo|transac[ao]\s+.{0,30}homologad", trecho):
        return "extinto acordo"
    if re.search(r"extint[ao]", trecho):
        return "extinto sem mérito"
    if re.search(r"indefiro\s+a\s+peticao\s+inicial|cancelamento\s+da\s+distribuicao", trecho):
        return "extinto sem mérito"
    return "não identificado"


# ── 11. culpa_atribuida ────────────────────────────────────────────────────────
_re_compart  = re.compile(
    r"\b(culpa\s+(concorrente|compartilhada|reciproca)|concorrencia\s+de\s+culpas?)\b",
    re.IGNORECASE
)
_re_terceiro = re.compile(
    r"\b(culpa\s+(exclusiva\s+)?de\s+terceiro|fraude\s+(praticada\s+)?por\s+terceiro"
    r"|estelionato\s+(praticado\s+)?por\s+terceiro)\b",
    re.IGNORECASE
)
_re_banco = re.compile(
    r"\b(falha\s+(na\s+)?prestacao\s+de\s+servicos?"
    r"|responsabilidade\s+.{0,20}(banco|requerido|reu)\s+.{0,20}reconhecida"
    r"|condeno\s+.{0,10}(banco|requerido|reu)"
    r"|dano\s+causado\s+pelo\s+(banco|requerido|reu))\b",
    re.IGNORECASE
)
_re_consumidor = re.compile(
    r"\b(culpa\s+(exclusiva\s+)?d[oa]\s+(autor[a]?|requerente|consumidor[a]?)"
    r"|ausencia\s+de\s+(falha|vicio|defeito|conduta\s+ilicita)"
    r"|nao\s+h[ao]\s+(falha|vicio|defeito)\s+na\s+prestacao)\b",
    re.IGNORECASE
)

def regex_culpa(t: str) -> str:
    n = _norm(t)
    if _re_compart.search(n):   return "compartilhada"
    if _re_terceiro.search(n):  return "terceiro"
    if _re_banco.search(n):     return "banco"
    if _re_consumidor.search(n): return "consumidor"
    return "não identificado"


# ── 12 & 13. valores monetários ────────────────────────────────────────────────
def _extrair_valor(texto: str, campo_re: str) -> float:
    """Extrai R$ X no trecho de ±150 chars ao redor do campo indicado."""
    t = str(texto)
    for pos in [m.start() for m in re.finditer(campo_re, t, re.IGNORECASE)]:
        trecho = t[max(0, pos - 150): pos + 200]
        for v in re.findall(r"R\$\s*([\d.,]+)", trecho):
            try:
                return float(v.replace(".", "").replace(",", "."))
            except ValueError:
                pass
    return 0.0

def regex_valor_morais(t: str) -> float:
    return _extrair_valor(t, r"danos\s+morais")

def regex_valor_materiais(t: str) -> float:
    return _extrair_valor(t, r"danos\s+materiais")


# ── 14. repetição_indébito ─────────────────────────────────────────────────────
_re_dobro   = re.compile(r"\b(repeticao\s+em\s+dobro|devolucao\s+em\s+dobro|indebito\s+em\s+dobro|restituicao\s+em\s+dobro)\b", re.IGNORECASE)
_re_simples = re.compile(r"\b(repeticao\s+do\s+indebito|restituicao\s+simples|devolucao\s+(do\s+)?valor|repeticao\s+simples)\b", re.IGNORECASE)

def regex_repeticao_indebito(t: str) -> str:
    n = _norm(t)
    if _re_dobro.search(n):   return "dobro"
    if _re_simples.search(n): return "simples"
    return "não identificado"


# ── Aplica tudo ────────────────────────────────────────────────────────────────
decisao = df_curto["decisao"].fillna("")

df_curto["tipo_documento_regex"]       = decisao.apply(regex_tipo_documento)
df_curto["fora_do_escopo_regex"]       = decisao.apply(regex_fora_do_escopo)
df_curto["justica_gratuita_regex"]     = decisao.apply(regex_justica_gratuita)
df_curto["rito_processual_regex"]      = decisao.apply(regex_rito_processual)
df_curto["tipo_acao_regex"]            = decisao.apply(regex_tipo_acao)
df_curto["contato_previo_banco_regex"] = decisao.apply(regex_contato_previo)
df_curto["canal_contato_regex"]        = decisao.apply(regex_canal_contato)
df_curto["mencao_reclame_aqui_regex"]  = decisao.apply(regex_mencao_reclame)
df_curto["boletim_de_ocorrencia_regex"] = decisao.apply(regex_boletim)
df_curto["resultado_julgamento_regex"] = decisao.apply(regex_resultado)
df_curto["culpa_atribuida_regex"]      = decisao.apply(regex_culpa)
df_curto["valor_danos_morais_regex"]   = decisao.apply(regex_valor_morais)
df_curto["valor_danos_materiais_regex"] = decisao.apply(regex_valor_materiais)
df_curto["repetição_indébito_regex"]   = decisao.apply(regex_repeticao_indebito)

print(f"Colunas _regex adicionadas: {len([c for c in df_curto.columns if c.endswith('_regex')])}")

In [ ]:
resultados_sem_cond = ["improcedente", "extinto sem mérito", 
                        "extinto com mérito prescrição", "extinto acordo"]

mask_sem_cond = df_curto["resultado_julgamento_ia"].isin(resultados_sem_cond)

df_curto.loc[
    mask_sem_cond & (df_curto["repetição_indébito_regex"] == "não identificado"),
    "repetição_indébito_regex"
] = "não aplicável"

Comparacao:

In [ ]:
## Comparação IA vs Regex

# Campos comparáveis (exclui valores numéricos — têm métrica própria)
campos_categoricos = [
    "tipo_documento",
    "justica_gratuita",
    "rito_processual",
    "tipo_acao",
    "contato_previo_banco",
    "canal_contato",
    "mencao_reclame_aqui",
    "boletim_de_ocorrencia",
    "resultado_julgamento",
    "culpa_atribuida",
    "repetição_indébito",
]

# Exclui linhas com ERRO na IA e fora do escopo
mask_validos = (
    ~df_curto["resultado_julgamento_ia"].astype(str).str.startswith("ERRO") &
    (df_curto["fora_do_escopo_ia"] == False)
)
sub = df_curto[mask_validos].copy()

print(f"Linhas analisadas (sem erro, sem fora do escopo): {len(sub)}\n")
print(f"{'CAMPO':<30} {'IGUAIS':>7} {'DIFER.':>7} {'ACORDO %':>9}")
print("-" * 58)

rows = []
for campo in campos_categoricos:
    col_ia    = f"{campo}_ia"
    col_regex = f"{campo}_regex"
    if col_ia not in sub.columns or col_regex not in sub.columns:
        continue

    ia    = sub[col_ia].astype(str).str.strip().str.lower()
    regex = sub[col_regex].astype(str).str.strip().str.lower()

    iguais = (ia == regex).sum()
    total  = len(sub)
    acordo = iguais / total * 100

    print(f"{campo:<30} {iguais:>7} {total - iguais:>7} {acordo:>8.1f}%")
    rows.append({"campo": campo, "iguais": iguais, "diferentes": total - iguais, "acordo_%": round(acordo, 1)})

df_concordancia = pd.DataFrame(rows).sort_values("acordo_%", ascending=False)

# Comparação de valores numéricos
print("\n--- Valores numéricos (só linhas com condenação > 0 em ambos) ---")
for campo_val in ["valor_danos_morais", "valor_danos_materiais"]:
    col_ia    = f"{campo_val}_ia"
    col_regex = f"{campo_val}_regex"
    ambos_pos = sub[(sub[col_ia] > 0) & (sub[col_regex] > 0)]
    if len(ambos_pos) == 0:
        print(f"{campo_val}: sem casos com ambos > 0")
        continue
    diff_pct = ((ambos_pos[col_ia] - ambos_pos[col_regex]).abs() / ambos_pos[col_ia] * 100)
    print(f"{campo_val}: n={len(ambos_pos)}, diferença média={diff_pct.mean():.1f}%, mediana={diff_pct.median():.1f}%")

# Inspeção manual das divergências
print("\n--- Inspeção: campos com menor acordo ---")
campo_pior = df_concordancia.iloc[-1]["campo"]
col_ia    = f"{campo_pior}_ia"
col_regex = f"{campo_pior}_regex"
divergentes = sub[sub[col_ia].astype(str).str.lower() != sub[col_regex].astype(str).str.lower()]
print(f"\nCampo '{campo_pior}' — {len(divergentes)} divergências. Amostra de 5:\n")
display(
    divergentes[["id_processo", col_ia, col_regex]]
    .head(5)
    .rename(columns={col_ia: "IA", col_regex: "Regex"})
)

In [ ]:
# Exportar xlsx com colunas _ia e _regex intercaladas

colunas_base = [c for c in df_curto.columns if not c.endswith("_ia") and not c.endswith("_regex")]

campos_ia = [c for c in df_curto.columns if c.endswith("_ia")]
campos_regex = [c.replace("_ia", "_regex") for c in campos_ia if c.replace("_ia", "_regex") in df_curto.columns]

colunas_intercaladas = []
for col_ia in campos_ia:
    colunas_intercaladas.append(col_ia)
    col_regex = col_ia.replace("_ia", "_regex")
    if col_regex in df_curto.columns:
        colunas_intercaladas.append(col_regex)

ordem_final = colunas_base + colunas_intercaladas
df_curto[ordem_final].to_excel("curto_ia_v2.xlsx", index=False)

print(f"Arquivo salvo: curto_ia_v2.xlsx")
print(f"Total colunas: {len(ordem_final)} ({len(colunas_base)} base + {len(colunas_intercaladas)} IA/regex)")
print("\nOrdem das colunas IA/regex:")
for i in range(0, len(colunas_intercaladas), 2):
    ia = colunas_intercaladas[i]
    rx = colunas_intercaladas[i+1] if i+1 < len(colunas_intercaladas) else "—"
    print(f"  {ia:<35} | {rx}")

## 3) Estatísticas das features

Resumo de distribuições e cruzamentos das variáveis extraídas.

In [ ]:
print("Distribuição da feature tem_justica_gratuita:")
print(df_curto["justica_gratuita_ia"].value_counts(dropna=False))

In [ ]:
coluna_assunto = "assunto" if "assunto" in df_curto.columns else "assuntos"
coluna_flag = "tem_justica_gratuita"

resumo_assunto = pd.crosstab(df_curto[coluna_assunto], df_curto[coluna_flag], dropna=False)
resumo_assunto = resumo_assunto.rename(columns={False: "nao", True: "sim"})

for c in ["sim", "nao"]:
    if c not in resumo_assunto.columns:
        resumo_assunto[c] = 0

resumo_assunto["total"] = resumo_assunto["sim"] + resumo_assunto["nao"]
resumo_assunto["pct_sim"] = (resumo_assunto["sim"] / resumo_assunto["total"] * 100).round(2)
resumo_assunto = resumo_assunto.sort_values(["sim", "total"], ascending=False)

print("Resumo por assunto (justiça gratuita: sim/nao):")
display(resumo_assunto.head(30))

print("\nAssuntos com pelo menos 1 ocorrência de JUSTIÇA GRATUITA:")
display(resumo_assunto[resumo_assunto["sim"] > 0].head(30))

print("\nAssuntos sem ocorrência de JUSTIÇA GRATUITA:")
display(resumo_assunto[resumo_assunto["sim"] == 0].head(30))

# Análise Exploratória de Dados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configurações de estilo
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## Mapas Coroplético — Processos por Comarca (Estado de SP)

Visualização geográfica dos processos sobre o mapa do estado de São Paulo.
Cada município é colorido de acordo com a métrica selecionada.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Carregar GeoJSON dos municípios de SP
with open(Path('geodata') / 'SP.json', 'r', encoding='utf-8') as f:
    sp_geojson = json.load(f)

# Normalizar nomes para matching
def _normalizar_nome_geo(nome):
    if pd.isna(nome): return ''
    t = unicodedata.normalize('NFKD', str(nome)).encode('ascii', 'ignore').decode('ascii')
    return t.strip().lower()

# Mapear nome do município -> GEOCODIGO
comarca_map = {}
for feat in sp_geojson['features']:
    comarca_map[_normalizar_nome_geo(feat['properties']['NOME'])] = feat['properties']['GEOCODIGO']

# Lista completa de todos os municípios de SP (base sempre desenhada)
sp_all_geocodigos = [feat['properties']['GEOCODIGO'] for feat in sp_geojson['features']]
sp_all_nomes = [feat['properties']['NOME'] for feat in sp_geojson['features']]

# Agregar por comarca
df_mapa = df_curto.groupby('comarca').agg(
    total_processos=('id_processo', 'count'),
    valor_medio_morais=('valor_danos_morais_ia', 'mean'),
).reset_index()

# Taxa de procedência
def _calc_taxa(g):
    total = len(g)
    if total == 0: return 0.0
    favoraveis = g['resultado_julgamento_ia'].isin(['procedente', 'parcialmente procedente']).sum()
    return (favoraveis / total) * 100

taxa = df_curto.groupby('comarca').apply(_calc_taxa, include_groups=False).reset_index()
taxa.columns = ['comarca', 'taxa_procedencia']
df_mapa = df_mapa.merge(taxa, on='comarca', how='left')

# Match comarca -> geocodigo
df_mapa['comarca_norm'] = df_mapa['comarca'].apply(_normalizar_nome_geo)
df_mapa['geocodigo'] = df_mapa['comarca_norm'].map(comarca_map)
df_mapa['valor_medio_morais'] = df_mapa['valor_medio_morais'].round(2)
df_mapa['taxa_procedencia'] = df_mapa['taxa_procedencia'].round(1)

# Estatísticas de match
matched = df_mapa['geocodigo'].notna().sum()
total_comarcas = len(df_mapa)
pct = matched / total_comarcas * 100
print(f'Comarcas mapeadas: {matched}/{total_comarcas} ({pct:.1f}%)')
if total_comarcas - matched > 0:
    sem_match = df_mapa[df_mapa['geocodigo'].isna()]['comarca'].tolist()
    print(f'Sem match ({total_comarcas - matched}):', sem_match)

df_plot = df_mapa[df_mapa['geocodigo'].notna()].copy()


def make_sp_choropleth(df, color_col, color_scale, title, value_label):
    # Camada 1 — base: todos os municípios de SP em cinza escuro neutro
    base = go.Choropleth(
        geojson=sp_geojson,
        locations=sp_all_geocodigos,
        featureidkey='properties.GEOCODIGO',
        z=[0] * len(sp_all_geocodigos),
        colorscale=[[0, '#1f2937'], [1, '#1f2937']],
        showscale=False,
        marker_line_color='rgba(255,255,255,0.35)',
        marker_line_width=0.4,
        hovertext=sp_all_nomes,
        hovertemplate='<b>%{hovertext}</b><br><i>sem processos</i><extra></extra>',
    )

    # Camada 2 — dados: municípios com valores na escala selecionada
    data_layer = go.Choropleth(
        geojson=sp_geojson,
        locations=df['geocodigo'],
        featureidkey='properties.GEOCODIGO',
        z=df[color_col],
        colorscale=color_scale,
        marker_line_color='white',
        marker_line_width=0.9,
        customdata=df[['comarca', 'total_processos', 'valor_medio_morais', 'taxa_procedencia']].values,
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'Processos: %{customdata[1]:,}<br>'
            'Danos morais médios: R$ %{customdata[2]:,.2f}<br>'
            'Procedência: %{customdata[3]:.1f}%'
            '<extra></extra>'
        ),
        colorbar=dict(title=value_label, thickness=14, len=0.75, outlinewidth=0),
    )

    fig = go.Figure(data=[base, data_layer])
    fig.update_geos(
        fitbounds='geojson',
        visible=False,
        projection_type='mercator',
        bgcolor='rgba(0,0,0,0)',
    )
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor='left'),
        margin={'r': 0, 't': 50, 'l': 0, 'b': 0},
        height=620,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
    )
    return fig


In [ ]:
# Mapa 1: Volume de processos por comarca
fig1 = make_sp_choropleth(
    df_plot,
    color_col='total_processos',
    color_scale='YlOrRd',
    title='Volume de Processos por Comarca — Estado de São Paulo',
    value_label='Nº Processos',
)
fig1.show()


In [ ]:
# Mapa 2: Valor médio de danos morais por comarca
fig2 = make_sp_choropleth(
    df_plot,
    color_col='valor_medio_morais',
    color_scale='Purples',
    title='Valor Médio de Danos Morais por Comarca — Estado de São Paulo',
    value_label='Valor Médio (R$)',
)
fig2.show()


In [ ]:
# Mapa 3: Taxa de procedência por comarca
fig3 = make_sp_choropleth(
    df_plot,
    color_col='taxa_procedencia',
    color_scale='Greens',
    title='Taxa de Procedência por Comarca — Estado de São Paulo',
    value_label='Procedência (%)',
)
fig3.show()


### 1. Proporção de Contato Prévio

## Qual a porcentagem geral de processos em que o autor tentou resolver o problema antes de judicializar?

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_curto, x='contato_previo_banco_ia', palette='viridis')
plt.title('Proporção de Casos com Contato Prévio ao Banco', fontsize=14)
plt.xlabel('Contato Prévio?', fontsize=12)
plt.ylabel('Quantidade de Processos', fontsize=12)

# Adicionar porcentagens acima das barras
total = len(df_curto['contato_previo_banco_ia'].dropna())
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height/total:.1%}', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=11)
plt.show()


### 2. Canais de Contato Mais Utilizados

## Quando o cliente procura o banco, por onde ele vai?

In [ ]:
canais = df_curto[df_curto['canal_contato_ia'] != 'não identificado']

plt.figure(figsize=(10, 5))
ax = sns.countplot(data=canais, y='canal_contato_ia', order=canais['canal_contato_ia'].value_counts().index, palette='magma')
plt.title('Canais de Contato Prévio Mais Utilizados', fontsize=14)
plt.xlabel('Quantidade de Processos', fontsize=12)
plt.ylabel('Canal de Contato', fontsize=12)
plt.show()


### 3. Visão Geral dos Resultados e Valores

## Como se distribuem os resultados das sentenças e as indenizações?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Resultados
sns.countplot(data=df_curto, y='resultado_julgamento_ia', 
              order=df_curto['resultado_julgamento_ia'].value_counts().index, 
              palette='coolwarm', ax=axes[0])
axes[0].set_title('Distribuição dos Resultados dos Julgamentos', fontsize=14)
axes[0].set_xlabel('Quantidade', fontsize=12)
axes[0].set_ylabel('Resultado', fontsize=12)

# Valores de Danos Morais (apenas > 0)
danos_morais_positivos = df_curto[df_curto['valor_danos_morais_ia'] > 0]
sns.histplot(danos_morais_positivos['valor_danos_morais_ia'], bins=15, kde=True, color='purple', ax=axes[1])
axes[1].set_title('Distribuição dos Valores de Danos Morais (Condenações > R$ 0)', fontsize=14)
axes[1].set_xlabel('Valor (R$)', fontsize=12)
axes[1].set_ylabel('Frequência', fontsize=12)

plt.tight_layout()
plt.show()


### 4. Contato Prévio vs. Probabilidade de Vitória

## Clientes que tentaram resolver administrativamente têm taxa de procedência maior?

In [ ]:
contato_resultado = pd.crosstab(df_curto['contato_previo_banco_ia'], df_curto['resultado_julgamento_ia'], normalize='index') * 100

ax = contato_resultado.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2')
plt.title('Resultado do Julgamento por Contato Prévio (%)', fontsize=14)
plt.xlabel('Houve Contato Prévio?', fontsize=12)
plt.ylabel('Proporção (%)', fontsize=12)
plt.legend(title='Resultado', bbox_to_anchor=(1.05, 1), loc='upper left')

# Adicionar os valores nas barras
for c in ax.containers:
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 0 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10)
    
plt.tight_layout()
plt.show()


### 5. Contato Prévio vs. Valores de Indenização (Danos Morais)

## Juízes aplicam indenizações maiores quando o cliente comprova que tentou contato?

In [ ]:
plt.figure(figsize=(10, 6))
# Filtrar apenas quem ganhou danos morais
condenados_morais = df_curto[df_curto['valor_danos_morais_ia'] > 0]

sns.boxplot(data=condenados_morais, x='contato_previo_banco_ia', y='valor_danos_morais_ia', palette='Set3')
plt.title('Distribuição do Valor de Danos Morais por Contato Prévio (Apenas Condenações > 0)', fontsize=14)
plt.xlabel('Houve Contato Prévio?', fontsize=12)
plt.ylabel('Valor de Danos Morais (R$)', fontsize=12)
plt.show()

# Imprimir as médias
medias = condenados_morais.groupby('contato_previo_banco_ia')['valor_danos_morais_ia'].mean()
print("Média de Danos Morais (quando há condenação):")
print(medias)


### 6. Impacto do Canal de Contato no Resultado

## Acionar o Procon/Reclame Aqui gera maior probabilidade de vitória do que o SAC?

In [ ]:
plt.figure(figsize=(10, 6))
canal_resultado = pd.crosstab(canais['canal_contato_ia'], canais['resultado_julgamento_ia'], normalize='index') * 100

sns.heatmap(canal_resultado, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': 'Proporção (%)'})
plt.title('Probabilidade de Resultado por Canal de Contato Prévio', fontsize=14)
plt.ylabel('Canal de Contato', fontsize=12)
plt.xlabel('Resultado do Julgamento', fontsize=12)
plt.show()


### 7. Tipo de Ação vs. Contato Prévio

## Quais os tipos de ação com maior volume de tentativa prévia de acordo?

In [ ]:
plt.figure(figsize=(10, 6))
tipo_contato = pd.crosstab(df_curto['tipo_acao_ia'], df_curto['contato_previo_banco_ia'], normalize='index') * 100

ax = tipo_contato.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Pastel1')
plt.title('Proporção de Contato Prévio por Tipo de Ação (%)', fontsize=14)
plt.xlabel('Tipo de Ação', fontsize=12)
plt.ylabel('Proporção (%)', fontsize=12)
plt.legend(title='Houve Contato?', bbox_to_anchor=(1.05, 1), loc='upper left')

for c in ax.containers:
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 0 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10)
    
plt.tight_layout()
plt.show()


### 8. Boletim de Ocorrência vs Probabilidade de Vitória

## Fazer B.O. ajuda a ganhar a causa?

In [ ]:
plt.figure(figsize=(10, 6))
bo_resultado = pd.crosstab(df_curto['boletim_de_ocorrencia_ia'], df_curto['resultado_julgamento_ia'], normalize='index') * 100

ax = bo_resultado.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Spectral')
plt.title('Resultado do Julgamento por Existência de B.O. (%)', fontsize=14)
plt.xlabel('Fez Boletim de Ocorrência?', fontsize=12)
plt.ylabel('Proporção (%)', fontsize=12)
plt.legend(title='Resultado', bbox_to_anchor=(1.05, 1), loc='upper left')

for c in ax.containers:
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 0 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10)
    
plt.tight_layout()
plt.show()


### 9. Atribuição de Culpa vs Contato Prévio

## Se houve contato, a culpa cai mais em cima do banco?

In [ ]:
plt.figure(figsize=(10, 6))
culpa_contato = pd.crosstab(df_curto['culpa_atribuida_ia'], df_curto['contato_previo_banco_ia'], normalize='columns') * 100

ax = culpa_contato.T.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='tab20')
plt.title('Distribuição de Culpa Atribuída de Acordo com Contato Prévio (%)', fontsize=14)
plt.xlabel('Houve Contato Prévio?', fontsize=12)
plt.ylabel('Proporção (%)', fontsize=12)
plt.legend(title='Culpa Atribuída', bbox_to_anchor=(1.05, 1), loc='upper left')

for c in ax.containers:
    labels = [f'{v.get_height():.1f}%' if v.get_height() > 0 else '' for v in c]
    ax.bar_label(c, labels=labels, label_type='center', fontsize=10)
    
plt.tight_layout()
plt.show()


### 10. Tabela Resumo (Relatório 2)

In [ ]:
# a) Porcentagem de processos com contato prévio
pct_contato = (df_curto['contato_previo_banco_ia'] == 'sim').mean() * 100

# b) Médias no subgrupo que fez contato prévio
subgrupo_contato = df_curto[df_curto['contato_previo_banco_ia'] == 'sim']

media_materiais_com_zeros = subgrupo_contato['valor_danos_materiais_ia'].mean()
media_morais_com_zeros = subgrupo_contato['valor_danos_morais_ia'].mean()

subgrupo_materiais_condenados = subgrupo_contato[subgrupo_contato['valor_danos_materiais_ia'] > 0]
subgrupo_morais_condenados = subgrupo_contato[subgrupo_contato['valor_danos_morais_ia'] > 0]

media_materiais_somente_condenados = subgrupo_materiais_condenados['valor_danos_materiais_ia'].mean()
media_morais_somente_condenados = subgrupo_morais_condenados['valor_danos_morais_ia'].mean()

print(f"a) Porcentagem com contato prévio: {pct_contato:.2f}%")
print(f"b) Média de danos materiais (incluindo zeros): R$ {media_materiais_com_zeros:.2f}")
print(f"c) Média de danos morais (incluindo zeros): R$ {media_morais_com_zeros:.2f}")
print(f"d) Média de danos materiais (somente condenados): R$ {media_materiais_somente_condenados:.2f}")
print(f"e) Média de danos morais (somente condenados): R$ {media_morais_somente_condenados:.2f}")

# Montar o dataframe e salvar no relatorio.xlsx
df_relatorio = pd.DataFrame([{
    "pct_contato_previo": pct_contato,
    "media_danos_materiais": media_materiais_com_zeros,
    "media_danos_morais": media_morais_com_zeros,
    "media_danos_materiais_somente_condenados": media_materiais_somente_condenados,
    "media_danos_morais_somente_condenados": media_morais_somente_condenados,
}])

df_relatorio.to_excel("relatorio.xlsx", index=False)
print("\nArquivo relatorio.xlsx gerado com sucesso!")

# Modelo Inferencial — Efeito do Contato Prévio com o Banco

**Três modelos inferenciais:**

| # | Modelo | Pergunta |
|---|--------|----------|
| A | Logit em **procedência** | O contato prévio aumenta a chance de o autor vencer? |
| B | OLS em **log(valor de danos morais)** | Em quem é condenado, o contato prévio muda o valor da condenação? |
| C | Logit em **contato prévio** | Qual o *perfil* de caso associado a ter havido contato prévio? |

Cada modelo é estimado em **3 especificações** (bivariada → com controles → com comarca) para isolar o efeito do contato dos confundidores (tipo de ação, rito, comarca, etc.). A estabilidade do coeficiente do contato entre as specs é o teste real de robustez da associação.


In [9]:
# === Configuração da análise inferencial ===
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np

RANDOM_STATE = 42

# Variáveis de interesse
VAR_TRATAMENTO = 'contato_previo_banco_ia'      # 'sim' / 'não'
NIVEL_BASE     = 'não'                           # referência para o coeficiente

# Controles do caso (não derivados da sentença)
CONTROLES_CASO = [
    'tipo_acao_ia',
    'rito_processual_ia',
    'boletim_de_ocorrencia_ia',
    'justica_gratuita_ia',
    'mencao_reclame_aqui_ia',
]

# Variáveis de alta cardinalidade — agrupadas (raras viram "outras")
ALTA_CARDIN = ['comarca', 'assunto']
MIN_FREQ_GRUPO = 20  # categorias com < N casos viram 'outras'

# Outcomes
OUTCOME_PROCED   = 'procedente'                 # binária derivada de resultado_julgamento_ia
OUTCOME_VL_MORAL = 'valor_danos_morais_ia'      # numérico (R$)


In [10]:
# === Preparação do dataset inferencial ===
df_inf = df_curto.copy()

# 1) Tratamento: contato prévio em 'sim'/'não'
df_inf = df_inf[df_inf[VAR_TRATAMENTO].isin(['sim', 'não'])].copy()

# 2) Outcome A — procedência (vitória do autor): procedente ou parcialmente procedente = 1
def _proced(r):
    if r in ['procedente', 'parcialmente procedente']: return 1
    if r in ['improcedente']:                          return 0
    return np.nan  # extinto / não identificado: fora desse modelo

df_inf[OUTCOME_PROCED] = df_inf['resultado_julgamento_ia'].apply(_proced)

# 3) Outcome B — log do valor de danos morais (somente em casos com condenação > 0)
df_inf['ln_morais']    = np.where(
    df_inf[OUTCOME_VL_MORAL] > 0,
    np.log(df_inf[OUTCOME_VL_MORAL].fillna(0).astype(float)),
    np.nan,
)
df_inf['condenado_moral'] = (df_inf[OUTCOME_VL_MORAL] > 0).astype(int)

# 4) Categóricas — NaN explícito + agrupar raras
for col in CONTROLES_CASO + [VAR_TRATAMENTO]:
    df_inf[col] = df_inf[col].fillna('desconhecido').astype(str)

for col in ALTA_CARDIN:
    df_inf[col] = df_inf[col].fillna('desconhecido').astype(str)
    counts = df_inf[col].value_counts()
    freq   = counts[counts >= MIN_FREQ_GRUPO].index
    df_inf[f'{col}_grp'] = df_inf[col].where(df_inf[col].isin(freq), 'outras')

print(f'Linhas totais: {len(df_curto):,}')
print(f'Linhas com tratamento válido (sim/não): {len(df_inf):,}\n')

print('Outcome A — procedência (descartado extinto/não identificado):')
print(df_inf[OUTCOME_PROCED].value_counts(dropna=False).rename('n'))
print()
print('Outcome B — condenado em danos morais > 0:')
print(df_inf['condenado_moral'].value_counts().rename('n'))
print()
print(f'Comarcas distintas após agrupamento: {df_inf["comarca_grp"].nunique()}')
print(f'Assuntos distintos após agrupamento: {df_inf["assunto_grp"].nunique()}')


NameError: name 'df_curto' is not defined

In [ ]:
# === Helpers para comparar especificações ===
def _termo_tratamento():
    """Nome do termo do tratamento dentro do output do statsmodels."""
    return f'C({VAR_TRATAMENTO}, Treatment("{NIVEL_BASE}"))[T.sim]'


def resumo_termo(modelo, termo, exponentiate=False):
    """Coef, std err, p-value, IC 95% para um termo específico.

    Se exponentiate=True (Logit), retorna OR e IC do OR.
    """
    if termo not in modelo.params.index:
        return None
    coef    = modelo.params[termo]
    se      = modelo.bse[termo]
    p       = modelo.pvalues[termo]
    ci_low, ci_high = modelo.conf_int().loc[termo]
    out = {
        'coef':       coef,
        'std_err':    se,
        'p_value':    p,
        'CI95_low':   ci_low,
        'CI95_high':  ci_high,
        'n_obs':      int(modelo.nobs),
    }
    if exponentiate:
        out.update({
            'OR':         np.exp(coef),
            'OR_CI95_lo': np.exp(ci_low),
            'OR_CI95_hi': np.exp(ci_high),
        })
    return out


def tabela_specs(modelos_dict, termo, exponentiate=False):
    """Tabela comparando o mesmo termo em várias especificações."""
    rows = []
    for nome, m in modelos_dict.items():
        r = resumo_termo(m, termo, exponentiate=exponentiate)
        if r is None:
            continue
        rows.append({'spec': nome, **r})
    df = pd.DataFrame(rows).set_index('spec')
    return df.round(4)


def estrela(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '.'
    return ''


In [ ]:
# === Modelo A — Logit em procedência ===
# H1: o contato prévio com o banco aumenta a probabilidade de o autor vencer?

df_a = df_inf.dropna(subset=[OUTCOME_PROCED]).copy()
df_a[OUTCOME_PROCED] = df_a[OUTCOME_PROCED].astype(int)

f_base    = f'{OUTCOME_PROCED} ~ C({VAR_TRATAMENTO}, Treatment("{NIVEL_BASE}"))'
f_controles = f_base + ' + ' + ' + '.join(f'C({c})' for c in CONTROLES_CASO)
f_completa  = f_controles + ' + C(comarca_grp) + C(assunto_grp)'

modelos_A = {
    '1_bivariado':  smf.logit(f_base,      data=df_a).fit(disp=0),
    '2_controles':  smf.logit(f_controles, data=df_a).fit(disp=0),
    '3_completo':   smf.logit(f_completa,  data=df_a).fit(disp=0),
}

print('Modelo A — Logit em procedência (vitória do autor)')
print(f'n = {len(df_a):,}\n')

tab_A = tabela_specs(modelos_A, _termo_tratamento(), exponentiate=True)
print('Efeito do contato prévio (vs "não") em cada especificação:')
print(tab_A)
print()
print('Leitura: OR > 1 → contato prévio aumenta chance de procedência; OR < 1 → reduz.')
print('Significância: *** p<0.001 | ** p<0.01 | * p<0.05 | . p<0.10')


In [ ]:
# === Modelo B — OLS em log(valor de danos morais), em condenados ===
# H2: entre quem é condenado em danos morais, o contato prévio muda o valor?

df_b = df_inf.dropna(subset=['ln_morais']).copy()

f_base      = f'ln_morais ~ C({VAR_TRATAMENTO}, Treatment("{NIVEL_BASE}"))'
f_controles = f_base + ' + ' + ' + '.join(f'C({c})' for c in CONTROLES_CASO)
f_completa  = f_controles + ' + C(comarca_grp) + C(assunto_grp)'

# HC3 = erros-padrão robustos a heterocedasticidade
modelos_B = {
    '1_bivariado':  smf.ols(f_base,      data=df_b).fit(cov_type='HC3'),
    '2_controles':  smf.ols(f_controles, data=df_b).fit(cov_type='HC3'),
    '3_completo':   smf.ols(f_completa,  data=df_b).fit(cov_type='HC3'),
}

print('Modelo B — OLS em log(valor de danos morais), somente condenações > 0')
print(f'n = {len(df_b):,}\n')

tab_B = tabela_specs(modelos_B, _termo_tratamento(), exponentiate=False)
print('Efeito do contato prévio (vs "não") em log(R$) em cada spec:')
print(tab_B)
print()
print('Leitura log-linear: % de mudança no valor ≈ (exp(coef) - 1) × 100.')


In [ ]:
# === Modelo C — Logit no contato prévio (perfil do caso) ===
# Qual o perfil de caso associado a ter havido contato prévio com o banco?

df_c = df_inf.copy()
df_c['contato_sim'] = (df_c[VAR_TRATAMENTO] == 'sim').astype(int)

# Aqui o contato é o OUTCOME — predito por características do caso
f_perfil = 'contato_sim ~ ' + ' + '.join(f'C({c})' for c in CONTROLES_CASO) + ' + C(comarca_grp) + C(assunto_grp)'
modelo_C = smf.logit(f_perfil, data=df_c).fit(disp=0)

# Top termos por |z-score| (significância da associação)
def top_termos(modelo, k=20, exclui_intercepto=True):
    df = pd.DataFrame({
        'coef':    modelo.params,
        'OR':      np.exp(modelo.params),
        'std_err': modelo.bse,
        'z':       modelo.tvalues,
        'p_value': modelo.pvalues,
    })
    df['sig'] = df['p_value'].apply(estrela)
    df['|z|'] = df['z'].abs()
    if exclui_intercepto and 'Intercept' in df.index:
        df = df.drop(index='Intercept')
    return df.sort_values('|z|', ascending=False).head(k).drop(columns='|z|').round(4)

print(f'Modelo C — Logit em contato_previo (n = {int(modelo_C.nobs):,})')
print(f'Pseudo R² (McFadden) = {modelo_C.prsquared:.4f}\n')
print('Top 20 características associadas a ter havido contato prévio:')
print(top_termos(modelo_C, k=20))


In [ ]:
# === Resumo de hipóteses — leitura inferencial direta ===

def relato_logit(modelos, termo, label_h):
    m = modelos['3_completo']
    r = resumo_termo(m, termo, exponentiate=True)
    if r is None:
        print(f'[{label_h}] termo não encontrado.')
        return
    s = estrela(r['p_value'])
    direc = 'aumenta' if r['OR'] > 1 else 'reduz'
    print(f'[{label_h}]')
    print(f'  OR = {r["OR"]:.2f}  (IC 95%: {r["OR_CI95_lo"]:.2f} – {r["OR_CI95_hi"]:.2f})')
    print(f'  p-value = {r["p_value"]:.4f} {s}')
    print(f'  Leitura: ter feito contato prévio {direc} as chances do desfecho em {abs(r["OR"]-1)*100:.1f}%, mantendo demais variáveis constantes.')
    print(f'  n = {r["n_obs"]:,}')
    print()


def relato_ols(modelos, termo, label_h, unidade='valor'):
    m = modelos['3_completo']
    r = resumo_termo(m, termo, exponentiate=False)
    if r is None:
        print(f'[{label_h}] termo não encontrado.')
        return
    s = estrela(r['p_value'])
    pct = (np.exp(r['coef']) - 1) * 100
    pct_lo = (np.exp(r['CI95_low']) - 1) * 100
    pct_hi = (np.exp(r['CI95_high']) - 1) * 100
    direc = 'maior' if pct > 0 else 'menor'
    print(f'[{label_h}]')
    print(f'  coef (log) = {r["coef"]:.3f}  (IC 95%: {r["CI95_low"]:.3f} – {r["CI95_high"]:.3f})')
    print(f'  p-value = {r["p_value"]:.4f} {s}')
    print(f'  Leitura: ter feito contato prévio está associado a um {unidade} {abs(pct):.1f}% {direc}, mantendo demais variáveis constantes (IC 95%: {pct_lo:+.1f}% a {pct_hi:+.1f}%).')
    print(f'  n = {r["n_obs"]:,}')
    print()


print('=' * 70)
print('CONCLUSÕES DO MODELO INFERENCIAL')
print('=' * 70)
print()

print('Pergunta central do plano: características do caso e da sentença')
print('associadas a processos com contato prévio ao banco?\n')

relato_logit(modelos_A, _termo_tratamento(),
             'H1 — Contato prévio → probabilidade de procedência')
relato_ols(modelos_B, _termo_tratamento(),
           'H2 — Contato prévio → valor da condenação (danos morais)',
           unidade='valor de condenação')

# H3 — perfil: cite os termos mais associados (z-score)
print('[H3 — Perfil dos casos com contato prévio]')
print('  Top 5 características mais associadas (ranking por |z|):')
top5 = top_termos(modelo_C, k=5)
for idx, row in top5.iterrows():
    direc = '↑' if row['coef'] > 0 else '↓'
    print(f'    {direc} {idx:<60} OR={row["OR"]:.2f}  p={row["p_value"]:.4f} {row["sig"]}')


## Como iterar

- **Adicionar controles:** inclua colunas em `CONTROLES_CASO`. Reexecute as células do Modelo A, B e C.
- **Trocar a referência do tratamento:** mude `NIVEL_BASE` (ex.: `'sim'` para inverter sinal).
- **Mudar o ponto de corte do agrupamento de raros:** ajuste `MIN_FREQ_GRUPO`. Quanto maior, menos categorias entram no modelo.
- **Outras hipóteses:**
  - **Interação `contato_previo × tipo_acao`:** acrescente `+ C(contato_previo_banco_ia):C(tipo_acao_ia)` na fórmula completa e teste se o efeito do contato depende do tipo de ação (efeito heterogêneo).
  - **Modelo de dois estágios para o valor:** primeiro Logit em `condenado_moral` (probabilidade de ser condenado), depois OLS em quem foi condenado (este já está implementado). A combinação dá o efeito médio total.
- **Robustez:**
  - Trocar `cov_type='HC3'` por `cov_type='cluster', cov_kwds={'groups': df_b['comarca_grp']}` para erros-padrão clusterizados por comarca.
  - Estimar com `sm.GLM(..., family=sm.families.Binomial())` para comparar com a Logit.
- **Comparação formal de specs:** `m.compare_lr_test(m_menor)` para teste de razão de verossimilhança entre modelos aninhados.
